In [ ]:
# v2 bundle bootstrap -- locate the bundle root (the first ancestor holding
# both lib/ and data/), put lib/ on sys.path, arm the CPU-f64 evaluation
# convention of the sealed v2 records BEFORE the first jax import, print the
# environment receipt.  `ognrepro.blocked` is imported FIRST: it pins the
# single-thread BLAS env vars at import time, so it must run before numpy
# acquires its thread pool.
import os
import sys
import time

_d = os.path.abspath(os.getcwd())
for _ in range(8):
    if os.path.isdir(os.path.join(_d, "lib")) and os.path.isdir(os.path.join(_d, "data")):
        break
    _d = os.path.dirname(_d)
BUNDLE_ROOT = _d
if os.path.join(BUNDLE_ROOT, "lib") not in sys.path:
    sys.path.insert(0, os.path.join(BUNDLE_ROOT, "lib"))
os.environ.setdefault("OGNREPRO_ROOT", BUNDLE_ROOT)

from ognrepro import blocked as B           # pins single-thread BLAS, then numpy
from ognrepro import repro_common as rc

conv = rc.enable_eval_convention()          # JAX_PLATFORMS=cpu + x64 + matmul 'highest'
assert os.path.samefile(rc.BUNDLE_ROOT, BUNDLE_ROOT), (rc.BUNDLE_ROOT, BUNDLE_ROOT)

import json
import numpy as np

SMOKE = os.environ.get("V2NB_SMOKE", "1") not in ("0", "false", "False", "")
def knob(name, smoke_value, full_value):
    v = os.environ.get(name)
    return int(v) if v else (smoke_value if SMOKE else full_value)

VALUES = os.path.join(BUNDLE_ROOT, "data", "values")
PROV   = os.path.join(BUNDLE_ROOT, "data", "provenance")
CKPT   = os.path.join(BUNDLE_ROOT, "checkpoints")

def load_values(name):
    with open(os.path.join(VALUES, name if name.endswith(".json") else name + ".json")) as fh:
        return json.load(fh)

def cite(doc, label=""):
    """Print the record@sha provenance block of an authored v2 sidecar."""
    prov = doc.get("provenance") or []
    print("  provenance%s: %d sealed record citation(s)" % ((" " + label) if label else "", len(prov)))
    for c in prov[:6]:
        print("    %-42s %-34s %s.." % (c["record"], c["file"], (c["file_sha256"] or "")[:16]))
    if len(prov) > 6:
        print("    ... %d more" % (len(prov) - 6))
    return prov

print("bundle root    :", BUNDLE_ROOT)
print("eval convention:", conv)
print("SMOKE          :", SMOKE, "(V2NB_SMOKE=0 for full size)")
_ = rc.print_env()

# Notebook A (v2) — the v2 configuration, the pinned stream, and HIGHEST-precision data

This is the **v2** distribution of the notebook release
(`fermionicml-notebooks-v2`).  It ships the same physics as v1 — the reduced
pairing (Richardson-type) Hamiltonian at fixed particle number, the inverse
map from the pair-scattering block $\rho^{(2)}_{ab}=\langle P_a^\dagger P_b\rangle$
and the total energy $E$ back to the couplings $G_{ab}$ — but a **different
production configuration**, carried behind one versioned object
(`ognrepro.v2.config.V2Config`, `cfg_version = 2`):

1. **HIGHEST-precision data.** Both the Hamiltonian build and the pair-block
   contraction of the dataset generator run at `jax.lax.Precision.HIGHEST`
   (`data_precision="highest"`), and the cache carries a covariance /
   ground-state sidecar.  The v1-era caches were generated with the default
   TPU matmul precision, i.e. bf16-rounded Hamiltonians — a label
   inconsistency the exploration branch measured at $\sim 4\times10^{-3}$,
   which is *above* the v2 clean medians and therefore had to go before the
   v2 numbers could mean anything.
2. **The exact affine readout on the dense-random family**
   (`readout="affine"`, `ognrepro.readout.project_affine`).  Two facts that
   are *linear in $G$* — the generator's diagonal gauge
   $\mathrm{mean}_k G_{kk} = 0.55$ and the energy-shell identity
   $\langle G,\rho\rangle_F = \sum_k \varepsilon^{\rm pair}_k \rho_{kk} - E$ —
   are imposed in closed form by a parameter-free $2\times2$ solve on top of
   the network's symmetric output.  The flax parameter tree is unchanged.
3. **The covariance-based ("state-aware metric") loss with a log-linear
   $\lambda$ anneal** (`loss="metric"`): 
   $L = \mathrm{mean}\big(q_M + \lambda(t)\,q_{S_\infty}\big) + w_{\rm ridge}\|\delta w\|^2$,
   with $q_M = \mathrm{Var}_{\rho_{\rm true}}(\delta H)$ and
   $q_{S_\infty} = \delta w^{\!\top}(S-uu^{\!\top})\delta w$, and
   $\lambda$ annealed log-linearly from `lam0` to `lam1` over
   `anneal_frac` of the run.

With every v2 flag off (`V2Config.published()`) the code paths *are* the
published ones, which is what the `v1chk_*` control lane in this bundle
exercises.

Sections: 1. the registry snapshot and what this dist carries · 2. the pinned
seed-1007 evaluation stream · 3. HIGHEST-precision exact-ED features ·
4. the d16 blocked-era reference surfaces.

## 1. The registry snapshot and the bundled roster

In [ ]:
# The frozen registry snapshot travels with the dist; every bundled lane must
# appear in it with a matching final_state sha256 (T0-grade: bytes on disk).
import hashlib

def sha256_file(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for b in iter(lambda: fh.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

with open(os.path.join(BUNDLE_ROOT, "tools", "copy_lists",
                       "V2-CHECKPOINTS.snapshot.json")) as fh:
    SNAP = json.load(fh)
by_lane = {e["lane"]: e for e in SNAP}
print("registry snapshot: %d lanes" % len(SNAP))

rows = []
for tier in ("d20", "d16prog"):
    d = os.path.join(CKPT, tier)
    for lane in sorted(os.listdir(d)):
        if not os.path.isdir(os.path.join(d, lane)):
            continue
        with open(os.path.join(d, lane, "config.json")) as fh:
            cfg = json.load(fh)
        e = by_lane[lane]
        rows.append((tier, lane, cfg["v2config"]["data_precision"],
                     cfg["v2config"]["readout"], cfg["v2config"]["loss"],
                     cfg["v2config"]["aux"], cfg["spec"]["state_type"],
                     cfg["spec"]["h_type"], e["n_params"], e["final_state_sha256"]))
print()
print("%-8s %-32s %-9s %-9s %-7s %-6s %-8s %-7s %10s  %s"
      % ("tier", "lane", "data_prec", "readout", "loss", "aux", "state", "h_type", "n_params", "final_state sha"))
print("-" * 140)
for r in rows:
    print("%-8s %-32s %-9s %-9s %-7s %-6s %-8s %-7s %10d  %s.."
          % (r[0], r[1], r[2], r[3], r[4], str(r[5]), r[6], r[7], r[8], r[9][:16]))
print("-" * 140)
print("%d lanes bundled out of %d in the registry snapshot" % (len(rows), len(SNAP)))

In [ ]:
# T0: re-hash every bundled final_state.msgpack and compare with the snapshot
# AND with the frozen pull table.  Nothing here trusts a manifest.
pull = {}
with open(os.path.join(BUNDLE_ROOT, "tools", "copy_lists", "checkpoints_v2.tsv")) as fh:
    for line in fh:
        line = line.rstrip("\n")
        if not line or line.startswith("#"):
            continue
        tier, lane, fname, sha, nbytes, uri = line.split("\t")
        pull[(lane, fname)] = (sha, int(nbytes), uri, tier)

t0 = time.time()
n_ck = 0
for tier, lane, *_ in rows:
    p = os.path.join(CKPT, tier, lane, "final_state.msgpack")
    got = sha256_file(p)
    assert got == by_lane[lane]["final_state_sha256"], (lane, "snapshot")
    assert got == pull[(lane, "final_state.msgpack")][0], (lane, "pull table")
    assert os.path.getsize(p) == pull[(lane, "final_state.msgpack")][1], (lane, "bytes")
    assert pull[(lane, "final_state.msgpack")][2].startswith("gs://"), lane
    n_ck += 1
print("[T0] %d bundled final_state.msgpack re-hashed == registry snapshot == frozen pull table  PASS (%.0f s)"
      % (n_ck, time.time() - t0))
print("[T0] epoch_state.msgpack in the bundle:",
      sum(1 for dp, dn, fn in os.walk(CKPT) for f in fn if f == "epoch_state.msgpack"),
      "(must be 0 -- only final states are distributed)")

## 2. The pinned seed-1007 evaluation stream

In [ ]:
# Every HONEST v2 clean number in this bundle is computed on the SAME rows:
# the seed-1007 validation stream, rows 0..4095, whose f32 buffer hashes to
# the pin below.  The sealed records declare that pin; the bundle ships the
# rows; the notebook re-hashes them.
LABELS_PIN = "94f616fb7c5d6b725fc0ba6bf91da346ca4498d901925ff854a090e1620ac011"
stream_p = os.path.join(BUNDLE_ROOT, "data", "streams", "v2_val1007_labels_first4096.npy")
labels = np.ascontiguousarray(np.asarray(np.load(stream_p), np.float32))
got = hashlib.sha256(labels[:4096].tobytes()).hexdigest()
print("stream       :", os.path.relpath(stream_p, BUNDLE_ROOT), labels.shape, labels.dtype)
print("rows sha256  :", got)
print("declared pin :", LABELS_PIN)
assert got == LABELS_PIN, "the bundled stream is not the pinned one"
print("[T0] pinned seed-1007 val rows 0..4095  PASS")

clean = load_values("v2_clean_medians.json")
cite(clean, "(v2_clean_medians)")
print("\nsealed convention:", clean["convention"])
print("pre-registered ratio band:", clean["preregistered_ratio_band"])
assert clean["labels_rows_sha256_pin"] == LABELS_PIN
n_pin = sum(1 for v in clean["lanes"].values() if v.get("labels_rows_match_pin"))
print("sealed lanes declaring labels_rows_match_pin: %d / %d"
      % (n_pin, len(clean["lanes"])))

In [ ]:
# The dense-random family reconstructs G from 55 labels (m=10, m(m+1)/2).
# The generator's diagonal gauge is 0.55 by construction -- an identity-grade
# property of the stream, checked here to 5e-7 (the campaign's own tolerance).
sys.path.insert(0, os.path.join(BUNDLE_ROOT, "lib"))
import expc1_cal1_archa1 as A          # sealed campaign-1 scoring primitives (vendored)

G = A.panel_matrices(labels[:1024])
dev = float(np.max(np.abs(G[:, np.arange(A.MP), np.arange(A.MP)].mean(axis=1) - A.DIAG_GAUGE_MEAN)))
print("panel  :", G.shape, "symmetric:", bool(np.array_equal(G, np.swapaxes(G, 1, 2))))
print("diag-mean gauge deviation: %.3e (tol %.1e)" % (dev, A.DIAG_GAUGE_TOL))
assert dev < A.DIAG_GAUGE_TOL
print("[T1] stream diagonal gauge == %.2f  PASS" % A.DIAG_GAUGE_MEAN)

## 3. HIGHEST-precision exact-ED features

The v2 evaluation convention is `exact64_cpu_highest`: the features the
network sees at score time are **rebuilt on the host in float64 by exact
diagonalisation** of the pair-restricted Hamiltonian (basis dimension
$\binom{10}{5}=252$ at $d_{\rm sp}=20$, $N=10$), never read back from a
served f32 cache.  `production_v2/stages/_common.features_exact64` is the
stage function; the same code is exercised here through the bundled engine.

The cell below rebuilds the features for a handful of rows, re-runs the first
and last row **serially** as a determinism gate, and checks the two exact
properties the pair block must satisfy.

In [ ]:
import scipy.linalg
import expc1_cpu64_companions as CC     # sealed campaign-1 f64 feature build (vendored)

N_FEAT = knob("V2NB_N_FEAT", 24, 256)
t0 = time.time()
engine = rc.get_engine()
engine.init_d20("random", "thermal", beta=1.0, g_init=A.G_INIT, g_stop=A.G_STOP)
print("engine bound in %.0f s; basis dimension D = %d" % (time.time() - t0, engine.basis.size))

rho2_dense = engine._safe_dense(engine.rho_2_kkbar_arrays)
ops_flat = np.asarray(rho2_dense, np.float64).reshape(-1, rho2_dense.shape[-1], rho2_dense.shape[-1])
print("pair-block operator set:", ops_flat.shape)

t0 = time.time()
BX = np.empty((N_FEAT, A.MP, A.MP), np.float64)
BE = np.empty((N_FEAT,), np.float64)
for i in range(N_FEAT):
    E, V = scipy.linalg.eigh(np.asarray(engine._shots_h_dense64(G[i]), np.float64))
    BX[i], BE[i] = CC.features_from_VE(V, E, ops_flat, beta=1.0)
print("f64 exact-ED features: %d rows in %.1f s" % (N_FEAT, time.time() - t0))

# determinism gate (the stage's own): rows 0 and N-1 recomputed serially
det = True
for i in (0, N_FEAT - 1):
    E, V = scipy.linalg.eigh(np.asarray(engine._shots_h_dense64(G[i]), np.float64))
    bx, be = CC.features_from_VE(V, E, ops_flat, beta=1.0)
    det = det and bool(np.array_equal(bx, BX[i]) and be == BE[i])
print("[T0] feature determinism (rows 0 and %d recomputed):" % (N_FEAT - 1), det, " PASS" if det else " FAIL")
assert det

sym = float(np.max(np.abs(BX - np.swapaxes(BX, 1, 2))))
print("[T1] pair block symmetry  max|rho - rho^T| = %.3e  PASS" % sym)
assert sym < rc.T1
print("     energy range over the sample: [%.6f, %.6f]" % (BE.min(), BE.max()))

In [ ]:
# Why HIGHEST matters -- replayed from the sealed data-precision study.
#
# The effect is a DEVICE effect: on TPU the default matmul precision runs the
# f32 contraction through bf16 passes, and it is that rounding which enters the
# labels.  A CPU backend has no bf16 path, so the difference CANNOT be
# reproduced here by re-running the contraction -- it has to be read off the
# sealed record that measured it.  V2-PRECISION-WAVE2 seals the exploration
# branch's wave-2 ladder, whose eval.json rows are CPU-f64 scores (n=4096) of
# lanes that differ ONLY in how their data were generated.
prec = load_values("v2_precision_values.json")
cite(prec, "(v2_precision)")
pr = prec["result"]
print("\nrecord disclosure:")
print("  " + pr["disclosure"][:400] + " ...")
print("  wave-2 report %s sha %s.." % (pr["report"]["path"], pr["report"]["sha256"][:16]))

ladder = pr["rows"]["ladder_rel_median"]
print("\nladder: median relative parameter error per generation variant (CPU-f64, n=4096)")
print("%-24s %s" % ("variant", "  ".join("%12s" % ("seed " + s) for s in sorted(ladder["base_gram"]))))
base = ladder["base_gram"]
for var in sorted(ladder):
    row = ladder[var]
    print("%-24s %s" % (var, "  ".join("%12.6e" % row[s] for s in sorted(row))))
print()
for var in sorted(ladder):
    if var == "base_gram":
        continue
    rat = [ladder[var][s] / base[s] for s in sorted(base) if s in ladder[var]]
    print("  %-24s x %.3f  %.3f  %.3f   (penalty vs the HIGHEST-generated baseline)"
          % (var, *rat))
    if var in ("base_gram_bf16data", "base_gram_feat"):
        assert min(rat) > 1.0, var

xe = pr["rows"]["cross_evals"]
print("\ncross-evaluations (same weights, validation features regenerated at a")
print("different precision):")
for k in sorted(xe):
    v = xe[k]
    print("  %-58s val %-8s rel_median %.6e" % (k, v["val_precision"], v["rel_median"]))
worst = max(v["rel_median"] for v in xe.values())
best_v2 = min(v["median_err_q"] for v in clean["lanes"].values()
              if v.get("state_type") == "thermal")
print("\nlargest sealed precision-induced deviation : %.3e" % worst)
print("best sealed v2 thermal clean median        : %.3e" % best_v2)
print("ratio                                      : %.1f x" % (worst / best_v2))
assert worst > best_v2, "the precision term must sit ABOVE the v2 clean medians"
print("\n[T0] the data-precision term is %.0fx the best v2 clean median -- which is why" % (worst / best_v2))
print("     the v2 caches are regenerated at HIGHEST before any v2 number is claimed  PASS")

with open(os.path.join(PROV, "v2_records_index.json")) as fh:
    IDX = json.load(fh)
assert "V2-PRECISION-WAVE2" in IDX["records"]
print("V2-PRECISION-WAVE2 sealed", IDX["records"]["V2-PRECISION-WAVE2"]["sealed_utc"],
      "manifest", IDX["records"]["V2-PRECISION-WAVE2"]["record_manifest_sha256"][:16], "..")

## 4. The d16 blocked-era reference surfaces

In [ ]:
# The d16 program (d_sp = 16, N = 8) is solved by the blocked f64 solver, and
# the v2 d16prog lanes are scored against the SEALED D16PROG-P1-REF target
# sets this bundle carries verbatim.
p1 = np.load(os.path.join(BUNDLE_ROOT, "data", "percells", "D16PROG-P1-REF",
                          "expc2r2_d16prog_p1_classical_percell.npz"))
for tag in ("ref_TS_U_1024", "ref_TS_V_512"):
    if tag + "_rho2" not in p1:
        continue
    rho2, en, lab = p1[tag + "_rho2"], p1[tag + "_energy"], p1[tag + "_labels"]
    s = float(np.max(np.abs(rho2 - np.swapaxes(rho2, 1, 2))))
    print("%-16s rho2 %s  energy %s  labels %s   max|rho-rho^T| = %.3e"
          % (tag, rho2.shape, en.shape, lab.shape, s))
    assert s < rc.T1
d16 = load_values("v2_d16score_values.json")
cite(d16, "(v2_d16score)")
print("\nsealed v2 d16prog lanes:", len(d16["lanes"]))
for lane in sorted(d16["lanes"])[:4]:
    v = d16["lanes"][lane]
    met = list(v["metrics"])[0]
    print("  %-32s %-9s %-11s n=%4d  median %s = %.6e"
          % (lane, v["sector"], v["target_set"], v["n"], met, v["metrics"][met]["median"]))
print("\nNotebook A done in %.0f s" % (time.time() - T0_NB) if "T0_NB" in dir() else "")

## 5. Where to next

* **Notebook B** restores the bundled v2 checkpoints and reproduces the sealed
  clean medians on exactly these rows (assertion class **T2**, and in fact
  bit-exact against the sealed per-sample prefix medians).
* **Notebook C** replays the sealed geometry, comparator and finite-shot
  records that the v2 claims rest on.
* **Notebook D** re-draws every v2 figure from its values sidecar and asserts
  bit-identity with the sealed payload (assertion class **T0**).